In [8]:
import pandas as pd
import numpy as np
import joblib

# Load the trained model from Phase 4
pipeline = joblib.load('../models/phase4_logistic_regression.pkl')
print("Model loaded successfully")

Model loaded successfully


In [9]:
def predict_churn_risk(customer_df):
    """
    Input: DataFrame with same columns as training data
    Output: DataFrame with churn_probability and risk_tier
    """
    proba = pipeline.predict_proba(customer_df)[:, 1]
    
    def tier(p):
        if p >= 0.7: return 'Critical'
        elif p >= 0.4: return 'High'
        elif p >= 0.2: return 'Medium'
        else: return 'Low'
    
    result = customer_df.copy()
    result['churn_probability'] = proba
    result['risk_tier'] = [tier(p) for p in proba]
    return result

In [6]:
def predict_churn_risk(customer_df):
    # Store ID if present, then drop for model
    ids = customer_df.get('customerID', None)
    
    # Drop ID if present (model never saw it)
    X = customer_df.drop('customerID', axis=1) if 'customerID' in customer_df.columns else customer_df.copy()
    
    proba = pipeline.predict_proba(X)[:, 1]
    
    def tier(p):
        if p >= 0.7: return 'Critical'
        elif p >= 0.4: return 'High'
        elif p >= 0.2: return 'Medium'
        else: return 'Low'
    
    result = X.copy()
    if ids is not None:
        result.insert(0, 'customerID', ids)
    result['churn_probability'] = proba
    result['risk_tier'] = [tier(p) for p in proba]
    return result

In [12]:
# Recreate the exact same data pipeline from Phases 2-4
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df = df.drop('customerID', axis=1)

X = df.drop('Churn', axis=1)
y = df['Churn'].map({'No': 0, 'Yes': 1})

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_test ready: {X_test.shape}")

X_test ready: (1407, 19)


In [14]:
# Simulate scoring "new" customers using test set
scored = predict_churn_risk(X_test.copy().reset_index(drop=True))

print(scored[['tenure', 'Contract', 'InternetService', 'churn_probability', 'risk_tier']].head(10))
print(f"\nRisk distribution:\n{scored['risk_tier'].value_counts()}")

   tenure        Contract InternetService  churn_probability risk_tier
0      59        Two year             DSL           0.051644       Low
1       7  Month-to-month     Fiber optic           0.783628  Critical
2      54        Two year              No           0.014391       Low
3       2  Month-to-month              No           0.400621      High
4      71        Two year     Fiber optic           0.232859    Medium
5      60  Month-to-month     Fiber optic           0.721028  Critical
6      33        One year              No           0.072998       Low
7       7  Month-to-month              No           0.348000    Medium
8      14  Month-to-month     Fiber optic           0.844323  Critical
9      72        Two year             DSL           0.047422       Low

Risk distribution:
risk_tier
Low         479
Critical    378
High        342
Medium      208
Name: count, dtype: int64


In [15]:
import os

# Create reports directory if needed
os.makedirs('../reports', exist_ok=True)

# Export critical customers for retention team
alerts = scored[scored['risk_tier'] == 'Critical'][['churn_probability', 'tenure', 'Contract', 'InternetService']]
alerts.to_csv('../reports/critical_alerts.csv', index=False)
print(f"{len(alerts)} critical alerts exported to ../reports/critical_alerts.csv")

378 critical alerts exported to ../reports/critical_alerts.csv


In [16]:
def api_predict(customer_dict):
    """Simulate REST API endpoint"""
    df = pd.DataFrame([customer_dict])
    result = predict_churn_risk(df)
    return {
        'churn_probability': float(result['churn_probability'].iloc[0]),
        'risk_tier': result['risk_tier'].iloc[0]
    }

# Test with a high-risk profile
test_customer = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 2,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'No',
    'StreamingMovies': 'No',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 70.0,
    'TotalCharges': 140.0
}

print(api_predict(test_customer))

{'churn_probability': 0.8544653606751216, 'risk_tier': 'Critical'}
